# 03 · Filter & Rank — run the shared multi-layer engine

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 05** you are both its *author* (you implemented the layers) and a *user*
(you run it on a pool). The point of this notebook: import the **shared** engine, build `fp.Design`
objects from the pool, run `fp.run_pipeline(...)`, and `fp.report(...)` the survival-at-each-layer
funnel + ranked CSV (D3 part 1).

> **Do not fork the module into this project.** Iterate locally against `shared/filtering_pipeline.py`
> and PR improvements back (notebook 05). This notebook *imports* it.

Run `00`–`02` first so `results/pool.csv` exists.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone.

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)

## Build `Design` objects from the pool

The engine operates on `fp.Design` records. Map each pool row onto its fields. The synthetic pool
pre-populates the metrics; in a real campaign these come from your upstream predictions (Projects
01/03). We keep the hidden `truth` label in `extra` so notebook 04 can measure enrichment — the
engine itself never sees it.

In [ ]:
import os
if not os.path.exists("results/pool.csv"):
    # regenerate if a fresh session lost it (deterministic seed)
    import make_example_pool as mep
    mep.make_pool(200).to_csv("results/pool.csv", index=False)

pool = pd.read_csv("results/pool.csv")

DESIGN_FIELDS = ["scrmsd", "plddt", "plddt_catalytic", "pae_interaction",
                 "catalytic_geom_rmsd", "scrmsd_orthogonal", "solubility",
                 "rosetta_dG", "shape_complementarity", "md_rmsd"]

def row_to_design(r):
    kw = {f: (None if pd.isna(r[f]) else float(r[f])) for f in DESIGN_FIELDS}
    return fp.Design(design_id=str(r["design_id"]), sequence="M",
                     design_type=str(r["design_type"]),
                     extra={"truth": r["truth"]}, **kw)

designs = [row_to_design(r) for _, r in pool.iterrows()]
print(len(designs), "Design objects built;",
      "types:", pool["design_type"].value_counts().to_dict())

## Run the pipeline (per design type)

`run_pipeline()` applies the chosen layers in order and returns a ranked DataFrame with survival
counts in `df.attrs`. The pool is **mixed**, and cutoffs differ by type, so we run each design type
with its own cutoffs, then concatenate. We use Layers 1–3 here (Layer 4 / MD is optional and added in
notebook 05).

In [ ]:
import pandas as pd

frames = []
survival_by_type = {}
for dt, sub in pool.groupby("design_type"):
    sub_designs = [row_to_design(r) for _, r in sub.iterrows()]
    df_dt = fp.run_pipeline(sub_designs, design_type=dt, use_layers=(1, 2, 3))
    survival_by_type[dt] = df_dt.attrs["survival"]
    df_dt["design_type"] = dt
    frames.append(df_dt)

ranked = pd.concat(frames, ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/pool_ranked.csv", index=False)
print("wrote results/pool_ranked.csv", ranked.shape)
print("\nsurvival by design type (L1->L3):")
for dt, s in survival_by_type.items():
    print(f"  {dt:9s} {s}")
ranked.head(10)[["design_id", "design_type", "layers_passed", "score",
                 "scrmsd", "plddt", "scrmsd_orthogonal", "solubility"]]

## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **whole
pool** treated with monomer cutoffs for a single, comparable funnel figure (the per-type run above is
the rigorous version). Read the bars as a funnel: steep drops show which layer discriminates; a layer
that cuts nothing is too lenient or redundant.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab will still display inline

all_designs = [row_to_design(r) for _, r in pool.iterrows()]
df_all = fp.run_pipeline(all_designs, design_type="monomer", use_layers=(1, 2, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p05")
print("\nsaved results/p05_survival.png + results/p05_ranked.csv")
top

## Honest survival accounting

How many designs reach each depth? `layers_passed` records the deepest layer each design survived.
This is the standard cohort artifact — but remember it is *survival*, not *correctness*. Notebook 04
adds the labeled-pool **enrichment** view (how many survivors are actually good).

In [ ]:
print("layers_passed distribution (whole pool, monomer cutoffs):")
print(df_all["layers_passed"].value_counts().sort_index())
n = len(df_all)
surv = df_all.attrs["survival"]
print("\nfunnel:")
print(f"  total            {n}")
for layer, k in surv.items():
    print(f"  survived {layer}      {k}  ({100*k/n:.1f}%)")

## D3 (part 1) checklist
- [ ] `results/pool_ranked.csv` produced by the **shared** module (not a one-off script).
- [ ] Survival-at-each-layer reported (funnel figure `results/p05_survival.png`).
- [ ] Per-design-type run done (each type with its own cutoffs).
- [ ] Mapping assumptions (which fields → which `Design` attributes) written down.

**Next:** `04_validate.ipynb` — the discrimination-problem analysis (enrichment + cutoff sensitivity).